# Distributed Systems Fundamentals

Notebook for **Chapter 12** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 Why Distributed, and What Can Go Wrong

In [ ]:
p=0.99
print("3 replicas, each up 99%: P(at least 2 up) =", round(3*p*p*(1-p)+p**3,6))
print("single node availability:", p)
print("5 replicas, majority = 3: P(>=3 up) =", round(sum(__import__('math').comb(5,k)*p**k*(1-p)**(5-k) for k in range(3,6)),8))
q=0.3
print("expected calls per request, max 3 tries, fail 30%:", round(1+q+q*q,2))
print("if server is down (100% fail), calls per request with 3 tries:", 3)
print("Little: L = lambda*W; 200 req/s * 0.05 s =", 200*0.05)
print("Pool of 10 threads, 0.05s each -> max throughput", 10/0.05, "req/s")

## 12.2 Replication and Consistency

In [ ]:
import random
# quorum replication: N=3 replicas, write to W, read from R
random.seed(1)
class Cluster:
    def __init__(s,n=3): s.rep=[(0,None)]*n  # (version,value)
    def write(s,ver,val,W):
        for i in random.sample(range(len(s.rep)),W): s.rep[i]=(ver,val)
    def read(s,R):
        got=[s.rep[i] for i in random.sample(range(len(s.rep)),R)]
        return max(got)
for W,R in [(1,1),(2,2),(3,1)]:
    stale=0
    for t in range(10000):
        c=Cluster(); c.write(1,"new",W)
        if c.read(R)[0]!=1: stale+=1
    print(f"N=3 W={W} R={R} W+R>N={W+R>3}  stale reads: {stale} of 10000")

## 12.3 Partitioning and Consistent Hashing

In [ ]:
import hashlib, bisect
def h(s): return int(hashlib.md5(s.encode()).hexdigest(),16)
def simple_assign(keys, n): 
    return {k: h(k)%n for k in keys}
keys=[f"user{i}" for i in range(1000)]
a=simple_assign(keys,4); b=simple_assign(keys,5)
moved=sum(a[k]!=b[k] for k in keys)
print("mod-N: keys moved going 4->5 nodes:",moved,"of 1000")

class Ring:
    def __init__(s,nodes,vnodes=100):
        s.ring=sorted((h(f"{n}#{v}"),n) for n in nodes for v in range(vnodes))
        s.hashes=[x[0] for x in s.ring]
    def get(s,k):
        i=bisect.bisect(s.hashes,h(k))%len(s.ring)
        return s.ring[i][1]
r4=Ring(["A","B","C","D"]); r5=Ring(["A","B","C","D","E"])
moved=sum(r4.get(k)!=r5.get(k) for k in keys)
print("ring: keys moved going 4->5 nodes:",moved,"of 1000")
from collections import Counter
print("ring load with 5 nodes:",sorted(Counter(r5.get(k) for k in keys).items()))

## 12.4 Consensus and Leader Election

In [ ]:
import random
random.seed(3)
# simplified leader election: highest alive id wins (bully); and raft-style majority vote
nodes={1:True,2:True,3:True,4:True,5:True}
def bully():
    return max(n for n,a in nodes.items() if a)
print("leader:",bully()); nodes[5]=False; print("5 crashed, new leader:",bully())
def majority(total,alive): return alive>total//2
for alive in (5,3,2):
    print(f"5-node cluster, {alive} alive -> can elect/commit? {majority(5,alive)}")
# raft-like election with random timeouts
def election():
    t={n:random.uniform(150,300) for n in range(1,6)}
    first=min(t,key=t.get)
    return first,round(t[first],1)
for term in range(1,4):
    print("term",term,"first candidate (node, timeout ms):",election())
# eventual consistency: last-write-wins merge
a={"x":(1,"red")}; b={"x":(2,"blue")}
merged={k:max(a.get(k,(0,)),b.get(k,(0,))) for k in {*a,*b}}
print("LWW merge:",merged)

## 12.5 CAP and PACELC

## 12.6 Retries, Timeouts, Idempotency and Backpressure

In [ ]:
import random
random.seed(7)
# retries: server fails 30% of calls
def call(p=0.3): return random.random()>p
def trial(max_try):
    for t in range(1,max_try+1):
        if call(): return True
    return False
for m in (1,2,3,5):
    ok=sum(trial(m) for _ in range(100000))
    print(f"max tries {m}: success {ok/1000:.1f}%   (theory {100*(1-0.3**m):.1f}%)")
# backoff with full jitter
def delays(n,base=0.1,cap=5):
    return [round(random.uniform(0,min(cap,base*2**i)),3) for i in range(n)]
print("full-jitter delays (s):",delays(6))
print("max possible per attempt:",[min(5,0.1*2**i) for i in range(6)])
# idempotency
class Bank:
    def __init__(s): s.bal=100; s.seen={}
    def pay(s,key,amt):
        if key in s.seen: return s.seen[key]
        s.bal-=amt; s.seen[key]="paid"; return "paid"
b=Bank()
for _ in range(3): b.pay("order-77",30)
print("with idempotency key, balance after 3 retries:",b.bal)
b2=100
for _ in range(3): b2-=30
print("without key, balance after 3 retries:",b2)

In [ ]:
# backpressure: bounded queue vs unbounded. arrivals 120/s, service 100/s, 10 s
def sim(limit):
    q=0;drop=0;lat=[]
    for sec in range(10):
        for _ in range(120):
            if limit is not None and q>=limit: drop+=1
            else: q+=1
        done=min(q,100); q-=done
    return q,drop
for lim in (None,50):
    q,d=sim(lim)
    print(f"limit={lim}: queue left={q}, rejected={d}, wait for last queued = {q/100:.2f}s")
# Little's law: L = lambda * W
lam=100; W=0.2
print("Little's law: throughput 100/s x 0.2s latency =",lam*W,"requests in flight")

## 12.7 Time, Order and Causality: Lamport and Vector Clocks

In [ ]:
def happened_before(a, b):
    return all(x <= y for x, y in zip(a, b)) and a != b

def relation(a, b):
    if happened_before(a, b): return "A before B"
    if happened_before(b, a): return "B before A"
    if a == b: return "same"
    return "concurrent"

# three replicas: [r0, r1, r2]
print(relation([2, 0, 0], [3, 1, 0]))
print(relation([3, 1, 0], [2, 0, 0]))
print(relation([2, 1, 0], [1, 0, 2]))
print(relation([1, 1, 1], [1, 1, 1]))

A before B
B before A
concurrent
same


## 12.8 Transactions Across Services: 2PC, Sagas and CRDTs

In [ ]:
def run_saga(steps, fail_at=None):
    done, log = [], []
    for i, (name, do, undo) in enumerate(steps):
        if i == fail_at:
            log.append(f"{name}: FAILED")
            for n, u in reversed(done):
                log.append(f"compensate {u}")
            return log
        log.append(f"{name}: ok")
        done.append((name, undo))
    return log

steps = [("reserve stock", None, "release stock"),
         ("charge card", None, "refund card"),
         ("create shipment", None, "cancel shipment")]
for line in run_saga(steps, fail_at=1): print(line)
print("---")
for line in run_saga(steps, fail_at=2): print(line)

reserve stock: ok
charge card: FAILED
compensate release stock
---
reserve stock: ok
charge card: ok
create shipment: FAILED
compensate refund card
compensate release stock


In [ ]:
def merge(x, y):
    return [max(a, b) for a, b in zip(x, y)]

A = [5, 0, 0]
B = [0, 3, 0]
C = [0, 0, 4]
left = merge(merge(A, B), C)
right = merge(C, merge(B, A))
again = merge(left, A)
print(left, right, again)
print("total likes:", sum(left))

[5, 3, 4] [5, 3, 4] [5, 3, 4]
total likes: 12


## 12.9 Raft Log Replication, Step by Step

Section 12.4 covered leader election. Once there is a leader, Raft replicates a log of commands. A command is **committed** once the leader has stored it on a majority of servers; only then is it applied and acknowledged to the client.

In [ ]:
for n in range(3, 8):
    majority = n // 2 + 1
    print(n, "servers: majority", majority, "tolerates", n - majority)

3 servers: majority 2 tolerates 1
4 servers: majority 3 tolerates 1
5 servers: majority 3 tolerates 2
6 servers: majority 4 tolerates 2
7 servers: majority 4 tolerates 3


## Chapter recap

## Practice set

Problem 12.7.1 (easy) How many nodes must be up in a 9-node Raft cluster to commit writes?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch12.html